In [ ]:
## Visualization - Matplotlib 3D Scatter Plot of all samples: compare methods
import matplotlib.pyplot as plt
import numpy as np
import torch
alias1 = '1220_pool'
alias2 = '1220_pool_conn'
#out_samples = torch.load("../base/test_dataset/preds_1208_conn.pt",weights_only=False)
out_samples1 = torch.load(f"../genconv/{alias1}/preds_{alias1}.pt",weights_only=False)
out_samples2 = torch.load(f"../genconv/{alias2}/preds_{alias2}.pt",weights_only=False)
sample = 34

#sc = ax.scatter(out_samples[sample]['nodes']['pos'],'k',alpha=0.3,label='initial position')
#ax[i,1].plot(out_samples[sample]['nodes']['pred_u'][:],alpha=0.3,label='predicted')

#ax.plot_trisurf(V[:,0],V[:,1],V[:,2],triangles=F,edgecolor='#888888',linewidth=lw,antialiased=True,color='#dddddd',alpha=0.1)

def n_nodes(sample):
    return sample['nodes']['pos'].shape[0]
map1 = {n_nodes(s): s for s in out_samples1}
map2 = {n_nodes(s): s for s in out_samples2}
keys = sorted(set(map1) & set(map2))

k = keys[sample]  # or pick any key you want
s1 = map1[k]
s2 = map2[k]

vis_scale = 50

def process(s):
    pos_init = s['nodes']['pos']
    pos_true = pos_init + s['nodes']['y_u'] * vis_scale
    pos_pred = pos_init + s['nodes']['pred_u'] * vis_scale
    pos_diff = np.linalg.norm(s['nodes']['y_u'], axis=-1)
    pos_diff_pred = np.linalg.norm(s['nodes']['pred_u'], axis=-1)
    mask = pos_diff > 1e-6
    pos_delta = np.mean(np.abs(pos_diff_pred[mask] - pos_diff[mask]) / pos_diff[mask]) if mask.any() else 0.0
    return pos_true, pos_pred, pos_diff, pos_diff_pred, pos_delta

pos_true1, pos_pred1, pos_diff1, pos_diff_pred1, pos_delta1 = process(s1)
pos_true2, pos_pred2, pos_diff2, pos_diff_pred2, pos_delta2 = process(s2)

fig, axes = plt.subplots(1, 3, figsize=(10, 5))

panels = [
    (axes[0], pos_true1, pos_diff1, pos_diff1, f'True Displacement (n={k})', pos_delta1),
    (axes[1], pos_pred1, pos_diff1, pos_diff_pred1, f'{alias1} (n={k})', pos_delta1),
    (axes[2], pos_pred2, pos_diff2, pos_diff_pred2, f'{alias2} (n={k})', pos_delta2),
]

for ax, pos_pred, pos_diff, pos_diff_pred, title, pos_delta in panels:
    sc = ax.scatter(pos_pred[:,1], pos_pred[:,2], s=10, marker='s', c=pos_diff_pred, cmap='jet', alpha=0.2)
    ax.set_title(f'{title} | Error: {pos_delta:.4f}')
    ax.axis('equal')

fig.colorbar(sc, ax=axes, fraction=0.03, pad=0.04)
plt.tight_layout()

C:\Users\bm_tu\AppData\Local\Temp\ipykernel_5600\1702736616.py:56: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
